# Upload Dataset to HuggingFace Hub

This notebook loads the `eng_dataset.csv` file (disease classification dataset with `label` and `text` columns), validates it, and uploads it to HuggingFace Hub using the `datasets` library.

## 1. Install and Import Required Libraries

In [1]:
%pip install datasets huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from datasets import Dataset, DatasetDict
from huggingface_hub import login
from getpass import getpass

/home/notlath/Documents/Thesis/aill-be-sick/notebooks/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure HuggingFace Authentication

Log in to HuggingFace Hub using your API token. The cell below uses `getpass` so your token is never displayed in the notebook output. Paste your token when prompted.

In [3]:
# Securely input your HuggingFace token (will not be displayed)
hf_token = getpass("Enter your HuggingFace API token: ")
login(token=hf_token)

## 3. Load the Local Dataset

Load `eng_dataset.csv` which contains disease classification data with two columns:
- **label**: The disease name (Dengue, Influenza, Measles, Pneumonia, Typhoid, Diarrhea)
- **text**: Patient symptom descriptions

In [4]:
df = pd.read_csv("fil_dataset.csv")
print(f"Loaded dataset with {len(df)} rows")
df.head(10)

Loaded dataset with 3000 rows


,label,text
0,Influenza,"Hindi ako nakapasada ngayon, bigla kasi akong ..."
1,Dengue,Nagsimula po ito na parang trangkaso lang. Aft...
2,Pneumonia,Medyo kaya pa naman kumilos pero may fever ako...
3,Pneumonia,Hindi ako makatulog nang maayos kung hindi nak...
4,Typhoid,Anim na araw na akong may sakit. Nagsimula sa ...
5,Measles,Hindi nawawala ang napakataas kong lagnat nito...
6,Pneumonia,"Sobrang tagal na nito, Doc, mag-two weeks na. ..."
7,Diarrhea,Mga pitong beses na akong nagtatae ngayong ara...
8,Typhoid,Pang-limang araw na 'to. Nagsimula sa mababang...
9,Typhoid,Hindi na ako makagalaw sa sobrang hina. Labing...


## 4. Explore and Validate the Dataset

Check shape, column types, class distribution, and missing values.

In [5]:
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nClass distribution:\n{df['label'].value_counts()}")

Shape: (3000, 2)

Column types:
label    str
text     str
dtype: object

Missing values:
label    0
text     0
dtype: int64

Class distribution:
label
Influenza    500
Dengue       500
Pneumonia    500
Typhoid      500
Measles      500
Diarrhea     500
Name: count, dtype: int64


In [6]:
# Preview a sample from each class
df.groupby("label").apply(lambda x: x.sample(1, random_state=42)).reset_index(drop=True)

,text
0,"Mainit ako since Tuesday, no cough or sipon, p..."
1,"Mga walong beses na akong nagtatae, at 'yung d..."
2,"I felt fine nung start ng shift, tapos biglang..."
3,"Isang linggong lagnat na ito, tapos sobrang sa..."
4,"Iho, tignan mo 'to, parang kulay prune juice o..."
5,Araw-araw tumataas ang temperatura ko sa loob ...


## 5. Create a HuggingFace Dataset Object

Convert the DataFrame to a HuggingFace `Dataset` and optionally split into train/test sets.

In [7]:

from datasets import ClassLabel, Features, Value

# Build label mapping
label_names = sorted(df["label"].unique().tolist())
label2id = {name: i for i, name in enumerate(label_names)}

# Encode labels as integers first (avoids ChunkedArray cast bug in datasets/PyArrow)
df_encoded = df.copy()
df_encoded["label"] = df_encoded["label"].map(label2id)

features = Features({
    "label": ClassLabel(names=label_names),
    "text": Value("string"),
})

# Convert pandas DataFrame to HuggingFace Dataset with typed features
dataset = Dataset.from_pandas(df_encoded, features=features)
print(dataset)

# Split into 70% train, 30% test (stratified)
dataset_dict = dataset.train_test_split(test_size=0.3, seed=42, stratify_by_column="label")
print(f"\nTrain: {dataset_dict['train'].num_rows} rows")
print(f"Test:  {dataset_dict['test'].num_rows} rows")


Dataset({
    features: ['label', 'text'],
    num_rows: 3000
})

Train: 2100 rows
Test:  900 rows


## 6. Push Dataset to HuggingFace Hub

Upload the dataset to your HuggingFace repository. Replace `"your-username/eng_dataset"` with your actual HuggingFace username and desired repo name.

In [8]:
# Set your HuggingFace repo name (format: "username/dataset-name")
REPO_NAME = "notlath/fil_dataset"

# Push the train/test split dataset to HuggingFace Hub
dataset_dict.push_to_hub(REPO_NAME, token=hf_token)

print(f"Dataset successfully uploaded to https://huggingface.co/datasets/{REPO_NAME}")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 76.73ba/s]
Processing Files (1 / 1): 100%|██████████|  298kB /  298kB,  115kB/s  
New Data Upload: 100%|██████████|  298kB /  298kB,  115kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 158.85ba/s]
Processing Files (1 / 1): 100%|██████████|  129kB /  129kB,  129kB/s  
New Data Upload: 100%|██████████|  129kB /  129kB,  129kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.95s/ shards]


Dataset successfully uploaded to https://huggingface.co/datasets/notlath/fil_dataset
